In [ ]:
import pandas as pd
import os
from datetime import datetime, timezone, timedelta

# Load Excel file
file_path = '240716__Literature Review_Analysis of Systems_final_HSF.xlsx'
df = pd.read_excel(file_path, sheet_name='Database final', header=0)

# Ensure directories for markdown files exist
os.makedirs('markdown_files', exist_ok=True)

def sanitize_field_name(name):
    """Sanitize field name to be a valid TOML key"""
    if isinstance(name, str):
        # Replace all spaces, parentheses, dashes, apostrophes, and other unwanted characters with underscores
        return name.lower() \
                   .replace(' ', '_') \
                   .replace('(', '') \
                   .replace(')', '') \
                   .replace('-', '_') \
                   .replace('–', '_') \
                   .replace('—', '_') \
                   .replace(',', '') \
                   .replace(';', '') \
                   .replace('/', '_') \
                   .replace("'", '')  # Remove apostrophes
    return "unknown_field"

def save_row_to_markdown_v2(row, index):
    # Initialize markdown content
    md_content = "+++\n"
    md_content += "date = \"" + str(datetime.now(timezone(timedelta(hours=1))).isoformat()) + "\"\n"
    md_content += "draft = false\n"

    system_name = row.get('Name', 'Default Name')
    if pd.isna(system_name) or system_name == "":
        system_name = "default_name"

    system_name_safe = "".join([c for c in system_name if c.isalnum() or c in [' ', '-', '_']]).replace(' ', '_')
    file_name = f"{index}_{system_name_safe}.md"

    for header, cell in row.items():
        if pd.isna(cell):
            cell = "N/A"
        field_name = sanitize_field_name(header)

        # Convert 'type_of_relationship', 'design_strategy', and 'tags' to arrays
        if header.lower() in ['type_of_relationship', 'design_strategy', 'tags']:
            if cell != "N/A":
                # Split by semicolon and strip extra whitespace, format as TOML array
                values = [f'"{v.strip()}"' for v in cell.split(';') if v.strip()]
                # Join the values in TOML array format
                md_content += f"{field_name} = [{', '.join(values)}]\n"
            else:
                md_content += f"{field_name} = []\n"  # Empty array if no values present

        # Handle other fields normally
        else:
            if isinstance(cell, str):
                cell = cell.replace('"', '\\"')  # Escape double quotes
            md_content += f'{field_name} = "{cell}"\n'

    md_content += "+++\n"
    md_path = os.path.join('markdown_files', file_name)
    with open(md_path, 'w', encoding='utf-8') as md_file:
        md_file.write(md_content)

    print(f"Generated Markdown for {file_name}")


# CSV Generation Function with UTF-8 encoding
def generate_csv_from_table(data, output_file):
    # Save the data into a CSV file with UTF-8 encoding
    data.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Generated CSV: {output_file}")

# Iterate through the DataFrame and save each row to a markdown file
for index, row in df.iterrows():
    save_row_to_markdown_v2(row, index)

# Generate CSV document for the selected columns with UTF-8 encoding
generate_csv_from_table(df[['Name', 'Publication Year', 'Publication Type', 'Reference']], "systems_table.csv")

print("Markdown files and CSV have been generated.")


In [ ]:
# Process CORE update file (250922_CORE_update.xlsx)
# Creates new .md files for papers, skipping description row

import pandas as pd
import os
from datetime import datetime, timezone, timedelta

# Load the update Excel file
# Row 2 = headers, Row 3 = descriptions (skip), Rows 4-22 = data (19 papers)
update_file_path = '250922_CORE_update.xlsx'
df_update = pd.read_excel(update_file_path, sheet_name=0, header=1, skiprows=[2], nrows=19)

# Starting index for new papers (update this based on existing papers)
start_index = 241

# Output directory - directly to content/papers
output_dir = 'content/papers'
os.makedirs(output_dir, exist_ok=True)

def sanitize_field_name(name):
    """Sanitize field name to be a valid TOML key"""
    if isinstance(name, str):
        return name.lower() \
                   .replace(' ', '_') \
                   .replace('(', '') \
                   .replace(')', '') \
                   .replace('-', '_') \
                   .replace('–', '_') \
                   .replace('—', '_') \
                   .replace(',', '') \
                   .replace(';', '') \
                   .replace('/', '_') \
                   .replace("'", '') \
                   .replace(':', '_')
    return "unknown_field"

def save_row_to_markdown_v2(row, index, output_dir):
    md_content = "+++\n"
    md_content += "date = \"" + str(datetime.now(timezone(timedelta(hours=1))).isoformat()) + "\"\n"
    md_content += "draft = false\n"

    system_name = row.get('Name', 'Default Name')
    if pd.isna(system_name) or system_name == "":
        system_name = "default_name"

    system_name_safe = "".join([c for c in system_name if c.isalnum() or c in [' ', '-', '_']]).replace(' ', '_')
    file_name = f"{index}_{system_name_safe}.md"

    for header, cell in row.items():
        if pd.isna(cell):
            cell = "N/A"
        field_name = sanitize_field_name(header)

        # Convert array fields
        if header.lower() in ['type_of_relationship', 'design_strategy', 'tags']:
            if cell != "N/A":
                values = [f'"{v.strip()}"' for v in str(cell).split(';') if v.strip()]
                md_content += f"{field_name} = [{', '.join(values)}]\n"
            else:
                md_content += f"{field_name} = []\n"
        else:
            if isinstance(cell, str):
                cell = cell.replace('"', '\\"')
            md_content += f'{field_name} = "{cell}"\n'

    md_content += "+++\n"
    md_path = os.path.join(output_dir, file_name)
    with open(md_path, 'w', encoding='utf-8') as md_file:
        md_file.write(md_content)

    print(f"Generated: {file_name}")

# Process each row from the update
for i, row in df_update.iterrows():
    new_index = start_index + i
    save_row_to_markdown_v2(row, new_index, output_dir)

print(f"\nGenerated {len(df_update)} new paper files (indices {start_index}-{start_index + len(df_update) - 1})")